# Calculating Climate Indicators on In-Situ Station Time Series

This notebook demonstrates how to use **earthkit-climate** to calculate climate indicators directly on **1D in-situ / station time series**.

While climate indicators are frequently computed over 2D spatial grids (such as ERA5 reanalysis or CMIP6 projections), observation networks and station datasets provide high-resolution, long-term point measurements. **earthkit-climate** indicator functions natively accept 1D and multi-station `xarray.DataArray` objects (e.g. dimensioned by `time` and `station_id`), automatically preserving station coordinates (`lat`, `lon`, `height`) and enriching the output with CF-compliant metadata.

---

## 1. Setup and Package Imports

First, we import `earthkit.data` for dataset loading, `earthkit.climate` for indicator computations, and standard scientific Python packages (`xarray`, `pandas`).

In [1]:
import earthkit.data as ekd
import pandas as pd

import earthkit.climate as ekc

## 2. Load Sample Station Dataset

We load a synthetic daily station temperature dataset provided by `earthkit-climate-sample`. This dataset contains daily maximum air temperature (`tasmax`) in Kelvin for 5 weather stations across a multi-year period (1991–2023).

In [2]:
# Load synthetic daily station dataset via earthkit.data
data_source = ekd.from_source("earthkit-climate-sample", "synthetic-daily-station-temperature")
ds_stations = data_source.to_xarray()

# Inspect dataset structure, dimensions, and coordinates
print("Dimensions:", ds_stations.dims)
print("Station IDs:", ds_stations.station_id.values)
print("Coordinates:", list(ds_stations.coords.keys()))
ds_stations

## 3. Inspect Station Coordinates and Metadata

Notice that the dataset has two core dimensions: `time` and `station_id`. Each station is associated with spatial metadata coordinates (`lat`, `lon`, and `height`/elevation).

In [3]:
# Display station metadata in tabular format
df_metadata = pd.DataFrame({
    "Station ID": ds_stations.station_id.values,
    "Latitude (°N)": ds_stations.lat.values,
    "Longitude (°E)": ds_stations.lon.values,
    "Elevation (m)": ds_stations.height.values,
})
df_metadata

## 4. Compute Climate Indicators on Station Data

We can now pass our station `DataArray` directly into **earthkit-climate** indicator functions.

### Example A: Hot Days Count (`tx_days_above`)
Count the annual number of hot days where daily maximum temperature exceeds 300 K (~26.85°C).

In [4]:
# Compute annual hot days for all stations
hot_days = ekc.indicators.tx_days_above(ds_stations, thresh="300 K", freq="YS")

# Check resulting dimensions and metadata
print("Hot Days output dimensions:", hot_days.dims)
print("CF Standard Name:", hot_days.attrs.get("standard_name"))
print("CF Cell Methods:", hot_days.attrs.get("cell_methods"))
hot_days

### Example B: Frost Days Count (`frost_days`)
Count the annual number of frost days where daily maximum/minimum temperature falls below freezing (0°C / 273.15 K).

In [5]:
# Compute annual frost days across stations
frost = ekc.indicators.frost_days(ds_stations, freq="YS")
frost

## 5. Station Summary & Tabular Inspection

Because station dimension coordinates (`station_id`, `lat`, `lon`, `height`) are preserved throughout the calculation, we can easily convert the result to a pandas DataFrame for analysis or reporting.

In [6]:
# Convert annual hot days indicator to a station-by-year table
df_hot_days = hot_days.to_dataframe().unstack(level="station_id")
df_hot_days.head(10)

## Summary

- **earthkit-climate** indicator functions work interchangeably on spatial gridded data and 1D station time series.
- Station dimension coordinates (`station_id`, `lat`, `lon`, `height`) and CF-compliant metadata are automatically retained across all reduction and indicator computations.